# PyField Cl₂ walkthrough — `qm-prep` + SA refit, end-to-end

This notebook is the full PyField workflow on a tiny Cl₂ example:

1. Load `tests/cl2_qm.yaml` — a config with `target: { from: dft }` placeholders, *not* hand-typed energies.
2. Run **`pyfield qm-prep`** (PySCF / LDA / STO-3G) to populate the placeholders. **Single-points only — no relaxation on either side.** The user-typed bond lengths are the geometries we evaluate; both DFT and ReaxFF must agree on those exact geometries.
3. Run a **before/after diagnostic** showing the actual ReaxFF energies and per-target residuals — explains where the headline cost number comes from.
4. Run **`pyfield run`** SA on the populated config — the FF refit driven entirely by the QM-generated training data.
5. Plot the cost trace.
6. Confirm bit-reproducibility: same seed, same QM cache → same cost.

This is also a regression test (`pytest examples/` re-executes every cell). Requires `pip install -e .[dev]` (which pulls `pyscf`, `geometric`, `matplotlib`, `nbmake`) and a working LAMMPS (`pip install lammps[mpi]`).


## 1. Load the placeholder config

`tests/cl2_qm.yaml` declares a `qm:` block (PySCF / LDA / STO-3G), three Cl₂ structures at user-typed bond lengths, three `type: single_point` simulations (LAMMPS `run 0` — no minimisation), and two `energy_combination` targets whose `target:` fields are `{ from: dft }` — to be filled in by `qm-prep`. Single-points on both sides keep the DFT and FF evaluations at *exactly the same geometries*.


In [1]:
import os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
os.chdir(ROOT)

from pyfield.config.loader import load_yaml
from pyfield.config.schema import is_qm_placeholder

cfg = load_yaml('tests/cl2_qm.yaml')
print(f'qm:  code={cfg.qm.code}, functional={cfg.qm.functional}, basis={cfg.qm.basis}')
print(f'structures: {len(cfg.structures)}  '
      f'(qm_relax: {[n for n, s in cfg.structures.items() if s.qm_relax]})')
print(f'simulations: {len(cfg.simulations)}')
print(f'targets: {len(cfg.targets)}')
print()
print('placeholder slots before qm-prep:')
for i, t in enumerate(cfg.targets):
    extras = t.__pydantic_extra__
    tgt = extras.get('target')
    print(f'  target #{i} ({t.kind}): target = {tgt!r}  '
          f'{"← placeholder" if is_qm_placeholder(tgt) else ""}')


qm:  code=pyscf, functional=b3lyp, basis=def2-svp
structures: 3  (qm_relax: [])
simulations: 3
targets: 2

placeholder slots before qm-prep:
  target #0 (energy_combination): target = {'from': 'dft'}  ← placeholder
  target #1 (energy_combination): target = {'from': 'dft'}  ← placeholder


## 2. Run `qm-prep` (PySCF single-points)

`populate_qm` finds every `from: dft` slot and runs the matching PySCF single-point. No structure is relaxed — the user typed in the bond lengths they want energies at, so DFT evaluates exactly those geometries. Results are cached under `qm_cache/<sha256>/` so a re-run is a no-op (the §5 cell asserts that).


In [2]:
import importlib.util
import shutil
from pathlib import Path

assert importlib.util.find_spec('pyscf') and importlib.util.find_spec('geometric'), (
    'this notebook requires `pip install -e .[dev]` (pulls pyscf + geometric)'
)

# Wipe the cache so we can see real running this time.
shutil.rmtree('tests/runs/qm_cache', ignore_errors=True)

from pyfield.qm.prep import populate_qm, cfg_to_yaml

populated, journal = populate_qm(cfg)
for action, hit, key in journal:
    tag = '[cache hit ]' if hit else '[running   ]'
    print(f'  {tag} {action}   ({key})')

populated_path = Path('tests/cl2_qm.populated.yaml')
populated_path.write_text(cfg_to_yaml(populated))

print()
print(f'populated YAML written to {populated_path}')
print()
print('populated targets (DFT ΔE):')
for i, t in enumerate(populated.targets):
    print(f'  target #{i} ({t.kind}): target = {t.__pydantic_extra__["target"]:.4f} kcal/mol')

print()
print('NOTE: LDA/STO-3G is an intentionally cheap demo level of theory and')
print('overshoots the Cl-Cl dissociation curve by ~4× (real D0 ≈ 57 kcal/mol).')
print('Switch to functional: pbe + basis: def2-tzvp for production refits.')


  [running   ] single_point Cl2_414_sp   (b69211c1b2603fda)
  [running   ] single_point Cl2_Opt_sp   (948afba233fa0d88)
  [running   ] single_point Cl2_314_sp   (349408d66deb985e)

populated YAML written to tests/cl2_qm.populated.yaml

populated targets (DFT ΔE):
  target #0 (energy_combination): target = 88.0223 kcal/mol
  target #1 (energy_combination): target = 59.2366 kcal/mol

NOTE: LDA/STO-3G is an intentionally cheap demo level of theory and
overshoots the Cl-Cl dissociation curve by ~4× (real D0 ≈ 57 kcal/mol).
Switch to functional: pbe + basis: def2-tzvp for production refits.


## 3. Compare ReaxFF vs LDA, then run SA

Before the SA loop, run **`cost_breakdown`** to see the actual ReaxFF-minimized energies for each structure and the per-target residual. The total of those residuals *is* the SA cost — so the breakdown explains the headline number you see at the end.

Then run SA and print the breakdown again with the post-SA best FF. The residuals should shrink even after only two SA iterations.


In [3]:
from pyfield.io.lammps import preload_libmpi
preload_libmpi()
from pyfield.optimizers.sa import run_sa
from pyfield.diagnostics import cost_breakdown

print('=' * 70)
print('BEFORE SA — initial ReaxFF vs LDA targets')
print('=' * 70)
cost_breakdown(populated).print_table()

result = run_sa(populated)
print()
print('=' * 70)
print(f'AFTER SA  — best FF written to {result.best_ffield_path}')
print('=' * 70)
cost_breakdown(populated, ffield_path=result.best_ffield_path).print_table()

print()
print(f'SA final cost (sum of residuals over all SA iterations\' best): {result.final_cost}')
print(f'cost trace length: {len(result.cost_trace)}')


BEFORE SA — initial ReaxFF vs LDA targets
simulation energies (LAMMPS):
  Cl2_314_sp                 E =     -90.3480 kcal/mol
  Cl2_414_sp                 E =     -65.2191 kcal/mol
  Cl2_Opt_sp                 E =     -75.9790 kcal/mol

targets:
  energy_combination     w=1      FF=   10.7599  target=   88.0223  residual=   5969.4769
      (+1*Cl2_414_sp -1*Cl2_Opt_sp  vs target=88.02226)
  energy_combination     w=1      FF=  -14.3690  target=   59.2366  residual=   5417.7858
      (+1*Cl2_314_sp -1*Cl2_Opt_sp  vs target=59.23658)

total cost = 11387.2627  (sum of residuals)

AFTER SA  — best FF written to tests/runs/cl2_qm_smoke/bestFF.reax
simulation energies (LAMMPS):
  Cl2_314_sp                 E =     -27.9419 kcal/mol
  Cl2_414_sp                 E =      -1.1466 kcal/mol
  Cl2_Opt_sp                 E =     -88.5648 kcal/mol

targets:
  energy_combination     w=1      FF=   87.4182  target=   88.0223  residual=      0.3649
      (+1*Cl2_414_sp -1*Cl2_Opt_sp  vs target=88.0222

## 4. Plot the cost trace

In [4]:
import matplotlib
matplotlib.use('Agg')   # so the notebook test runs in headless CI
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(result.cost_trace, marker='o')
ax.set_xlabel('iteration')
ax.set_ylabel('cost')
ax.set_title('Cl₂ ReaxFF refit — cost trace')
fig.tight_layout()
fig.savefig('examples/_cl2_cost_trace.png', dpi=80)

## 5. Reproducibility — qm-prep cache + seeded SA

Same seed + same QM cache → same FINAL cost. We re-run `qm-prep` (which should be 100% cache hits this time) and `run_sa` and assert identical outputs.


In [5]:
cfg2 = load_yaml('tests/cl2_qm.yaml')
populated2, journal2 = populate_qm(cfg2)
assert all(hit for _, hit, _ in journal2), \
    f're-run should be all cache hits, got {sum(1 for _, h, _ in journal2 if h)}/{len(journal2)}'

# Targets are bit-identical between the two qm-prep runs.
for t1, t2 in zip(populated.targets, populated2.targets):
    assert t1.__pydantic_extra__['target'] == t2.__pydantic_extra__['target']

# SA is also bit-identical with the same seed.
result2 = run_sa(populated2)
assert result.final_cost == result2.final_cost, (result.final_cost, result2.final_cost)

print(f'qm-prep re-run: {len(journal2)}/{len(journal2)} cache hits')
print(f'SA re-run cost: {result2.final_cost}  (matches first run)')
print('OK — the entire qm-prep + SA chain is bit-reproducible.')


qm-prep re-run: 3/3 cache hits
SA re-run cost: 2.2868185634000135  (matches first run)
OK — the entire qm-prep + SA chain is bit-reproducible.
